# HRNet-W32 joint fine-tuning

Fine-tune the ImageNet-1k HRNet-W32 body and the selected local linear head together on stored fold 0. Keep the 384-pixel letterbox and fixed 0.25 threshold. This notebook calls the experiment CLI so the same recovery and export code works outside Jupyter. The selected-fold score is a screening result, not an independent ensemble gain.


In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import time
import os
import random
import shutil
import subprocess
import sys

import numpy as np

SEED = 42
RUN_ID = "hrnet_w32_finetune_fold0_seed42"
MODEL_NAME = "hrnet_w32.ms_in1k"
FOLD = 0
IMAGE_SIZE = 384
THRESHOLD = 0.25
BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 16
MAX_EPOCHS = 20
PATIENCE = 4

# Colab: CUDA, fp16 mixed precision, inputs and backups on Drive.
# Local Mac: Apple GPU (MPS), full fp32 as in the other Mac runs, inputs from this repository, no Drive.
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    PRECISION = "16-mixed"  # Use "bf16-mixed" if the Colab GPU supports it.
    BUDGET_HOURS = 5.0  # Colab session limit, including setup, the profile epoch and export.
    DRIVE_ROOT = Path("/content/drive/MyDrive/AI301-stage2")
    INPUT_ROOT = DRIVE_ROOT / "hrnet_w32_input"
    DRIVE_OUTPUT = DRIVE_ROOT / "hrnet_w32_runs"
    WORK_ROOT = Path("/content/hrnet_w32_work")
else:
    PRECISION = "32-true"  # The fp16 path is CUDA-only.
    BUDGET_HOURS = 8.0  # Wall-clock limit for this notebook; the profile epoch below sets the epoch count from it.
    DRIVE_OUTPUT = None
    start = Path(os.environ.get("STAGE2_ROOT", Path.cwd())).resolve()
    WORK_ROOT = next((path for path in (start, *start.parents)
                      if (path / "folds.csv").is_file() and (path / "stage2" / "hrnet_finetune.py").is_file()), None)
    if WORK_ROOT is None:
        raise FileNotFoundError("Launch Jupyter from this repository or set STAGE2_ROOT")
HEAD_DIR = WORK_ROOT / "artifacts/hrnet_head/hrnet_w32_head_fold0_seed42"
RESUME_CHECKPOINT = None  # Set to a resume_epoch_XX.ckpt when recovering (Drive on Colab, artifacts/ locally).
EXPORT_ONLY = False  # Set True with RESUME_CHECKPOINT after training ended but export failed.

session_started = time.monotonic()
random.seed(SEED)
np.random.seed(SEED)
if EFFECTIVE_BATCH_SIZE % BATCH_SIZE:
    raise ValueError("Effective batch size must be divisible by physical batch size")
print(f"{'Colab' if IN_COLAB else 'Local'} run: {MODEL_NAME}, fold {FOLD}, 384 px, {PRECISION}, "
      f"batch {BATCH_SIZE} × {EFFECTIVE_BATCH_SIZE // BATCH_SIZE} accumulation, budget {BUDGET_HOURS:g} h")
print("Repository:", WORK_ROOT)


Local run: hrnet_w32.ms_in1k, fold 0, 384 px, 32-true, batch 2 × 8 accumulation, budget 8 h
Repository: /Users/allakovalcuk/AI301/deep-learning-assignment2


## Inputs

**Colab:** run `uv run --frozen python scripts/stage_hrnet_colab_inputs.py` locally and upload its `artifacts/hrnet_w32_input/` folder to `MyDrive/AI301-stage2/hrnet_w32_input/`. It contains the branch code, locked environment, dataset, selected local head, and source manifest. Frozen feature shards stay local. This is the only interactive authentication step. The run writes epoch backups and final artifacts under `hrnet_w32_runs/`.

**Local:** the repository itself is the workspace. The selected head must be in `artifacts/hrnet_head/hrnet_w32_head_fold0_seed42/` (`head_best.pt`, `head_best.json`, `backbone.json`) and the competition data in `data/`. Checkpoints and predictions stay under `artifacts/hrnet_finetune/`; there is no Drive copy. W&B uses the key in `.env` if present.


In [2]:
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    for required in (INPUT_ROOT / "repo/stage2/hrnet_finetune.py", INPUT_ROOT / "repo/uv.lock",
                     INPUT_ROOT / "repo/folds.csv", INPUT_ROOT / "data/train.csv",
                     INPUT_ROOT / "data/test.csv", INPUT_ROOT / "data/sample_submission.csv",
                     INPUT_ROOT / "head/head_best.pt", INPUT_ROOT / "head/head_best.json",
                     INPUT_ROOT / "head/backbone.json", INPUT_ROOT / "manifest.json"):
        if not required.is_file():
            raise FileNotFoundError(f"Missing Colab input: {required}")
    if not WORK_ROOT.exists():
        shutil.copytree(INPUT_ROOT / "repo", WORK_ROOT,
                        ignore=shutil.ignore_patterns(".git", ".venv", "artifacts", "__pycache__", ".env"))
        shutil.copytree(INPUT_ROOT / "data", WORK_ROOT / "data")
        shutil.copy2(INPUT_ROOT / "manifest.json", WORK_ROOT / "source_manifest.json")
        HEAD_DIR.mkdir(parents=True)
        for name in ("head_best.pt", "head_best.json", "backbone.json"):
            shutil.copy2(INPUT_ROOT / "head" / name, HEAD_DIR / name)
    else:
        print("Using existing runtime workspace:", WORK_ROOT)
    manifest = json.loads((WORK_ROOT / "source_manifest.json").read_text())
    for relative, expected_hash in manifest["files"].items():
        group, _, remainder = relative.partition("/")
        staged = (WORK_ROOT / remainder if group == "repo" else
                  HEAD_DIR / remainder if group == "head" else WORK_ROOT / "data" / remainder)
        digest = hashlib.sha256(staged.read_bytes()).hexdigest()
        if digest != expected_hash:
            raise ValueError(f"Staged input failed checksum: {staged}")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    try:
        wandb_key = userdata.get("WANDB_API_KEY")
    except Exception:
        wandb_key = None
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    print("Inputs staged; W&B online logging:", bool(wandb_key))
else:
    for required in (HEAD_DIR / "head_best.pt", HEAD_DIR / "head_best.json", HEAD_DIR / "backbone.json",
                     WORK_ROOT / "data/train.csv", WORK_ROOT / "data/test.csv", WORK_ROOT / "data/sample_submission.csv"):
        if not required.is_file():
            raise FileNotFoundError(f"Missing local input: {required}")
    print("Local inputs found; head:", HEAD_DIR)


Local inputs found; head: /Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_head/hrnet_w32_head_fold0_seed42


## Environment and preflight

Install the locked environment (Colab only; locally the repository's `.venv` is used), verify the fixed split and every image, then check one GPU optimizer update (CUDA on Colab, MPS on a Mac) and measure one complete epoch before fixing the full run schedule. A failed smoke test leaves no training run directory.


In [3]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "sync", "--frozen", "--python", "3.12"], cwd=WORK_ROOT, check=True)
base_command = ["uv", "run", "--frozen", "python", "-m", "stage2.hrnet_finetune",
                "--root", str(WORK_ROOT), "--head-dir", str(HEAD_DIR),
                "--run-id", RUN_ID, "--batch-size", str(BATCH_SIZE),
                "--max-epochs", str(MAX_EPOCHS), "--precision", PRECISION]
subprocess.run([*base_command, "--preflight-only"], cwd=WORK_ROOT, check=True)


Inputs verified: 9808 train, 2452 validation, 18673 test; head epoch 40; labels ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']


CompletedProcess(args=['uv', 'run', '--frozen', 'python', '-m', 'stage2.hrnet_finetune', '--root', '/Users/allakovalcuk/AI301/deep-learning-assignment2', '--head-dir', '/Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_head/hrnet_w32_head_fold0_seed42', '--run-id', 'hrnet_w32_finetune_fold0_seed42', '--batch-size', '2', '--max-epochs', '20', '--precision', '32-true', '--preflight-only'], returncode=0)

In [4]:
subprocess.run([*base_command, "--smoke-only"], cwd=WORK_ROOT, check=True)


Inputs verified: 9808 train, 2452 validation, 18673 test; head epoch 40; labels ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']


/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/allakovalcuk/AI301/deep-learning-assignment2/stage2/hrnet_finetune.py:160: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  f"loss {float(loss):.5f}, {time.perf_counter() - start:.1f}s, "


Smoke update on Apple MPS (arm64): batch 2, precision 32-true, loss 0.00037, 1.2s, GPU memory 1.49 GiB


CompletedProcess(args=['uv', 'run', '--frozen', 'python', '-m', 'stage2.hrnet_finetune', '--root', '/Users/allakovalcuk/AI301/deep-learning-assignment2', '--head-dir', '/Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_head/hrnet_w32_head_fold0_seed42', '--run-id', 'hrnet_w32_finetune_fold0_seed42', '--batch-size', '2', '--max-epochs', '20', '--precision', '32-true', '--smoke-only'], returncode=0)

## Choose a run length from measured throughput

A complete train and validation epoch measures this GPU and data path. On Colab the profile is archived to Drive. The notebook reserves 45 minutes for prediction export and allows 30% margin on each training epoch within `BUDGET_HOURS`, then sets the cosine schedule horizon before the actual run.


In [5]:
profile_path = WORK_ROOT / "artifacts/hrnet_profile" / f"{RUN_ID}.json"
if RESUME_CHECKPOINT is None:
    subprocess.run([*base_command, "--profile-epoch"], cwd=WORK_ROOT, check=True)
    profile = json.loads(profile_path.read_text())
    if IN_COLAB:
        drive_profile = DRIVE_OUTPUT / RUN_ID / "profile.json"
        drive_profile.parent.mkdir(parents=True, exist_ok=True)
        if not drive_profile.exists():
            shutil.copy2(profile_path, drive_profile)
        if hashlib.sha256(drive_profile.read_bytes()).hexdigest() != hashlib.sha256(profile_path.read_bytes()).hexdigest():
            raise IOError("Drive profile copy failed checksum verification")
    remaining_seconds = BUDGET_HOURS * 3600 - (time.monotonic() - session_started)
    available_training_seconds = remaining_seconds - 45 * 60
    planned_epochs = min(MAX_EPOCHS, math.floor(available_training_seconds / (1.30 * profile["full_epoch_seconds"])))
    if planned_epochs < 1:
        raise RuntimeError(f"This GPU cannot fit one joint-training epoch plus export in the remaining {BUDGET_HOURS:g}-hour budget")
    run_budget_hours = remaining_seconds / 3600
    print(f"Measured {profile['full_epoch_seconds'] / 60:.1f} min/epoch, "
          f"GPU memory {profile['peak_allocated_gib']:.2f} GiB; plan {planned_epochs} epochs")
else:
    record_dir = DRIVE_OUTPUT / RUN_ID / "results" if IN_COLAB else WORK_ROOT / "artifacts/hrnet_finetune" / RUN_ID
    previous = json.loads((record_dir / "run.json").read_text())
    planned_epochs = previous["config"]["max_epochs"]
    run_budget_hours = previous["config"]["budget_hours"]
    print(f"Resuming existing {planned_epochs}-epoch schedule")
base_command[base_command.index("--max-epochs") + 1] = str(planned_epochs)
base_command.extend(["--budget-hours", str(run_budget_hours)])


Seed set to 42


Inputs verified: 9808 train, 2452 validation, 18673 test; head epoch 40; labels ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.

Full epoch: 33.2 min; peak GPU memory 2.58 GiB; profile /Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_profile/hrnet_w32_finetune_fold0_seed42.json
Measured 33.2 min/epoch, GPU memory 2.58 GiB; plan 9 epochs


## Joint fine-tuning and export

The CLI starts from the local head, trains body and head with warm-up and cosine decay, keeps BatchNorm running statistics fixed, and selects the best epoch by fold-0 macro F1. Every completed epoch gets a full resume checkpoint and checksum-verified Drive copy. To resume after a VM reset, set `RESUME_CHECKPOINT` to a Drive `resume_epoch_XX.ckpt`; this cell restores the exact best checkpoint recorded in that resume state. Set `EXPORT_ONLY=True` if training has ended and only prediction export needs recovery. The planned horizon comes from the measured full epoch above. Validation and test CSVs contain independent sigmoid probabilities for all five labels; matching logits are saved separately. The source notebook is archived with results, while Colab saves the executed notebook through its normal Drive save. Nothing is submitted to Kaggle.


In [6]:
command = [*base_command, *(["--drive-dir", str(DRIVE_OUTPUT)] if IN_COLAB else [])]
if RESUME_CHECKPOINT is not None and not IN_COLAB:
    # Locally every checkpoint is already in artifacts/hrnet_finetune/<run>/checkpoints/.
    resume_local = Path(RESUME_CHECKPOINT)
    if not resume_local.is_file():
        raise FileNotFoundError(resume_local)
    command.extend(["--resume", str(resume_local)])
elif RESUME_CHECKPOINT is not None:
    drive_run = DRIVE_OUTPUT / RUN_ID
    local_run = WORK_ROOT / "artifacts/hrnet_finetune" / RUN_ID
    checkpoint_dir = local_run / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    resume_source = Path(RESUME_CHECKPOINT)
    if not resume_source.is_file():
        raise FileNotFoundError(resume_source)
    import torch
    resume_state = torch.load(resume_source, map_location="cpu", weights_only=False)
    selection = resume_state["callbacks"]["hrnet_selected_checkpoint"]
    best_source = drive_run / "checkpoints" / f"best_epoch_{selection['best_epoch']:02d}.ckpt"
    if not best_source.is_file():
        raise FileNotFoundError(f"Missing selected checkpoint for resume: {best_source}")
    def digest(path):
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").digest()
    def restore(source, destination):
        if source.resolve() == destination.resolve():
            return
        if destination.exists():
            if digest(source) != digest(destination):
                raise FileExistsError(f"Existing restored file differs: {destination}")
            return
        shutil.copy2(source, destination)
        if digest(source) != digest(destination):
            raise IOError(f"Restored file failed checksum: {destination}")
    restore(drive_run / "results/run.json", local_run / "run.json")
    restore(best_source, checkpoint_dir / best_source.name)
    resume_local = checkpoint_dir / resume_source.name
    restore(resume_source, resume_local)
    command.extend(["--resume", str(resume_local)])
if EXPORT_ONLY:
    if RESUME_CHECKPOINT is None:
        raise ValueError("EXPORT_ONLY needs RESUME_CHECKPOINT")
    command.append("--export-only")
subprocess.run(command, cwd=WORK_ROOT, check=True)


Seed set to 42


Inputs verified: 9808 train, 2452 validation, 18673 test; head epoch 40; labels ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.
wandb: Currently logged in as: hhavva (models-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: setting up run hrnet_w32_finetune_fold0_seed42
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_finetune/hrnet_w32_finetune_fold0_seed42/wandb/run-20261020_185925-hrnet_w32_finetune_fold0_seed42
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run hrnet_w32_finetune_fold0_seed42
wandb: ⭐️ View project at https://forge.coreweave.com/wandb/models-kyiv-school-of-economics/ai301-stage2
wandb: 🚀 View run at https://forge.coreweave.com/wandb/models-kyiv-school-of-economics/ai301-stage2/runs/hrnet_w32_finetune_fold0_seed42
GPU available

epoch 1: macro F1 0.86458; per-label F1 {'Cat': 0.9798, 'Dog': 0.9249, 'Moris': 0.6667, 'Motya': 0.8182, 'Biatrix': 0.9333}; macro AP 0.93713; train/val BCE 0.02174/0.03441; best epoch 1


/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/allakovalcuk/AI301/deep-learning-assignment2/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


epoch 9: macro F1 0.99723; per-label F1 {'Cat': 0.9951, 'Dog': 0.9911, 'Moris': 1.0, 'Motya': 1.0, 'Biatrix': 1.0}; macro AP 0.99991; train/val BCE 0.00011/0.00844; best epoch 8


`Trainer.fit` stopped: `max_epochs=9` reached.
wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary
wandb: 
wandb: Run history:
wandb:                        epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇███
wandb: gradient_norm_backbone_epoch █▆▅▄▃▂▁▁▁
wandb:  gradient_norm_backbone_step ▂▄▁█▃▂▁▂▂▃▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:          gradient_norm_epoch █▆▅▄▃▂▁▁▁
wandb:     gradient_norm_head_epoch █▆▅▄▃▂▁▁▁
wandb:      gradient_norm_head_step █▁▄▂▂▁▄▅▃▅▄▁▃▁▄▁▅▁▁▁▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:           gradient_norm_step █▁▄▂▆▃▂▁▅▄▂▄▁▃▁▂▂▁▃▁▁▁▁▁▂▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:                   train_loss ██▆▅▃▂▁▁▁
wandb:           train_loss_current ██▆▅▃▂▁▁▁
wandb:          trainer/global_step ▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
wandb:                          +18 ...
wandb: 
wandb: Run summary:
wandb:                        epoch 8
wandb: gradient_norm_backbone_epoch 0.04876
wandb:  gradient_norm_bac

Selected epoch 8; macro F1 0.99733; per-label F1 {'Cat': 0.9941176470588236, 'Dog': 0.9925187032418953, 'Moris': 1.0, 'Motya': 1.0, 'Biatrix': 1.0}; macro AP 0.99995; per-label AP {'Cat': 0.9999330982987393, 'Dog': 0.9998056878304271, 'Moris': 1.0, 'Motya': 1.0000000000000002, 'Biatrix': 1.0}; artifacts /Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_finetune/hrnet_w32_finetune_fold0_seed42


CompletedProcess(args=['uv', 'run', '--frozen', 'python', '-m', 'stage2.hrnet_finetune', '--root', '/Users/allakovalcuk/AI301/deep-learning-assignment2', '--head-dir', '/Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_head/hrnet_w32_head_fold0_seed42', '--run-id', 'hrnet_w32_finetune_fold0_seed42', '--batch-size', '2', '--max-epochs', '9', '--precision', '32-true', '--budget-hours', '7.402407540150283'], returncode=0)

## Results

Read the selected checkpoint record and compact per-label metrics. The CSVs in Drive are the inputs for local threshold and ensemble analysis.


In [7]:
run_dir = WORK_ROOT / "artifacts/hrnet_finetune" / RUN_ID
results = json.loads((run_dir / "results.json").read_text())
print("Selected epoch:", results["selected_epoch"])
print("Checkpoint SHA-256:", results["checkpoint_sha256"])
print(f"Macro F1: {results['metrics']['macro_f1']:.5f}; macro AP: {results['metrics']['macro_ap']:.5f}")
for label in ("Cat", "Dog", "Moris", "Motya", "Biatrix"):
    print(f"{label:8s} F1 {results['metrics']['per_label_f1'][label]:.5f}  AP {results['metrics']['per_label_ap'][label]:.5f}")
import csv
with (run_dir / "history.csv").open(newline="") as stream:
    selected_history = next(row for row in csv.DictReader(stream) if int(row["epoch"]) == results["selected_epoch"])
print("Selected-epoch validation BCE by label:",
      {label: round(float(selected_history[f"bce_{label}"]), 5) for label in ("Cat", "Dog", "Moris", "Motya", "Biatrix")})
print("Run directory:", run_dir)
if IN_COLAB:
    print("Drive:", DRIVE_OUTPUT / RUN_ID)
print("Probability files:", results["files"]["validation"]["probabilities"]["path"],
      results["files"]["test"]["probabilities"]["path"])


Selected epoch: 8
Checkpoint SHA-256: c8a8a36762400162cbfa3c439b7fee748091f7f7c133c0ead126b7fc297ec071
Macro F1: 0.99733; macro AP: 0.99995
Cat      F1 0.99412  AP 0.99993
Dog      F1 0.99252  AP 0.99981
Moris    F1 1.00000  AP 1.00000
Motya    F1 1.00000  AP 1.00000
Biatrix  F1 1.00000  AP 1.00000
Selected-epoch validation BCE by label: {'Cat': 0.01335, 'Dog': 0.0223, 'Moris': 0.00014, 'Motya': 7e-05, 'Biatrix': 0.0001}
Run directory: /Users/allakovalcuk/AI301/deep-learning-assignment2/artifacts/hrnet_finetune/hrnet_w32_finetune_fold0_seed42
Probability files: validation_probabilities.csv test_probabilities.csv


## Save the run in the experiments layout

The CLI keeps its working files and recovery checkpoints under `artifacts/hrnet_finetune/` (ignored by Git). This cell copies the finished run into the same layout as the other experiments, so the ensemble notebooks can read it like Exp1, Exp3 and Exp5:

- `experiments/exp07/results/<run>/<run>.json`: run record (configuration, fold-0 metrics, checkpoint hash, versions);
- `experiments/exp07/results/<run>/<run>_val.csv`: fold-0 probabilities, plus `<run>_val_logits.csv` and `history.csv` (per-epoch curves);
- `experiments/exp07/results/<run>_test/<run>_test.csv`: test probabilities, plus `<run>_test_logits.csv` and `<run>_test.json`;
- `experiments/exp07/results/<run>/checkpoints/best.ckpt`: the selected checkpoint (ignored by Git, as for Exp5).

The fold-0 metrics are re-scored from the copied CSV, and every copy is checked against the hashes recorded at export. Existing files are never overwritten. On Colab the folders are also copied to Drive.


In [8]:
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from stage2.experiment import LABEL_COLUMNS, score_probabilities


def file_hash(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def place(source: Path, destination: Path, expected: str | None = None, link: bool = False) -> None:
    """Copy (or hard-link) once; an existing destination must be identical."""
    source_hash = file_hash(source)
    if expected is not None and source_hash != expected:
        raise ValueError(f"{source} differs from the hash recorded at export")
    if destination.exists():
        if file_hash(destination) != source_hash:
            raise FileExistsError(f"Refusing to overwrite a different file: {destination}")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    os.link(source, destination) if link else shutil.copy2(source, destination)
    if file_hash(destination) != source_hash:
        raise IOError(f"Copy failed verification: {destination}")


def write_once(path: Path, record: dict) -> None:
    """Write once; on a rerun the record must match, except the public score entered by hand after upload."""
    text = json.dumps(record, indent=2, allow_nan=False) + "\n"
    if path.exists():
        saved, fresh = json.loads(path.read_text()), json.loads(text)
        saved.pop("public_leaderboard_score", None), fresh.pop("public_leaderboard_score", None)
        if saved != fresh:
            raise FileExistsError(f"Refusing to overwrite a different record: {path}")
        return
    path.write_text(text)


run_record = json.loads((run_dir / "run.json").read_text())
head_record = json.loads((HEAD_DIR / "head_best.json").read_text())
files = results["files"]
exp_results = WORK_ROOT / "experiments/exp07/results"
val_dir, test_dir = exp_results / RUN_ID, exp_results / f"{RUN_ID}_test"
val_csv, test_csv = val_dir / f"{RUN_ID}_val.csv", test_dir / f"{RUN_ID}_test.csv"
checkpoint_copy = val_dir / "checkpoints" / "best.ckpt"

place(run_dir / files["validation"]["probabilities"]["path"], val_csv, files["validation"]["probabilities"]["sha256"])
place(run_dir / files["validation"]["logits"]["path"], val_dir / f"{RUN_ID}_val_logits.csv", files["validation"]["logits"]["sha256"])
place(run_dir / "history.csv", val_dir / "history.csv")
place(run_dir / files["test"]["probabilities"]["path"], test_csv, files["test"]["probabilities"]["sha256"])
place(run_dir / files["test"]["logits"]["path"], test_dir / f"{RUN_ID}_test_logits.csv", files["test"]["logits"]["sha256"])
place(run_dir / "checkpoints" / Path(results["checkpoint"]).name, checkpoint_copy, results["checkpoint_sha256"], link=True)

# Re-score fold 0 from the copied file, so the record matches exactly what the ensemble notebooks will read.
with val_csv.open(newline="") as stream:
    val_rows = list(csv.DictReader(stream))
with (run_dir / files["validation"]["targets"]["path"]).open(newline="") as stream:
    target_rows = list(csv.DictReader(stream))
if [row["image_id"] for row in val_rows] != [row["image_id"] for row in target_rows]:
    raise ValueError("Validation probabilities and targets are not aligned")
val_probabilities = np.asarray([[float(row[label]) for label in LABEL_COLUMNS] for row in val_rows])
val_targets = np.asarray([[int(float(row[label])) for label in LABEL_COLUMNS] for row in target_rows])
at_threshold = score_probabilities(val_targets, val_probabilities, THRESHOLD)
at_half = score_probabilities(val_targets, val_probabilities, 0.5)
with (run_dir / "history.csv").open(newline="") as stream:
    epochs_completed = len(list(csv.DictReader(stream)))
with test_csv.open(newline="") as stream:
    test_rows_count = sum(1 for _ in csv.DictReader(stream))

config = run_record["config"]
relative = lambda path: str(path.relative_to(WORK_ROOT))
versions = {name: run_record[name] for name in ("python", "torch", "timm", "lightning")}
transform = "mean-colour letterbox, bicubic, ImageNet normalization"
write_once(val_dir / f"{RUN_ID}.json", {
    "run_id": RUN_ID, "experiment": "07_hrnet_w32", "status": "completed", "fold": FOLD, "seed": SEED,
    "model": {"name": MODEL_NAME, "variant": "standard", "family": "hrnet",
              "initial_head_checkpoint": "experiments/exp07/checkpoints/head_best.pt",
              "initial_head_sha256": run_record["head_checkpoint_sha256"],
              "head_warmup": {"notebook": "notebooks/16_hrnet_w32_head.ipynb", "epochs_completed": head_record["epoch"],
                              "best_heldout_macro_f1": head_record["macro_f1"]}},
    "data": {"image_size": IMAGE_SIZE, "transform": transform + "; train horizontal flip p=0.5",
             "fold_file": "folds.csv", "fold_sha256": file_hash(WORK_ROOT / "folds.csv"),
             "label_columns": list(LABEL_COLUMNS), **head_record["normalization"]},
    "training": {"optimizer": f"AdamW(backbone_lr={config['backbone_lr']}, head_lr={config['head_lr']}, "
                              f"weight_decay={config['weight_decay']} except bias/norm, betas=(0.9, 0.999))",
                 "scheduler": "one epoch step warmup then cosine to 0.1 peak", "loss": "BCEWithLogitsLoss",
                 "max_epochs": config["max_epochs"], "patience": config["patience"],
                 "physical_batch_size": config["batch_size"], "effective_batch_size": config["effective_batch_size"],
                 "precision": config["precision"],
                 "stage_policy": "full network and classifier trainable; BatchNorm running statistics frozen",
                 "checkpoint_selection": "best validation macro F1 at 0.25; ties by lower validation BCE",
                 "epochs_completed": epochs_completed, "selected_epoch": results["selected_epoch"],
                 "budget_hours": config["budget_hours"]},
    "git_commit": run_record["git_revision"], "hardware": {"device": run_record["gpu"]},
    "metrics": {**at_threshold, "macro_f1_at_0_5": at_half["macro_f1"], "per_label_f1_at_0_5": at_half["per_label_f1"]},
    "artifacts": {"validation_probabilities_csv": relative(val_csv),
                  "validation_logits_csv": relative(val_dir / f"{RUN_ID}_val_logits.csv"),
                  "history_csv": relative(val_dir / "history.csv"),
                  "best_checkpoint_local": relative(checkpoint_copy),
                  "best_checkpoint_sha256": results["checkpoint_sha256"],
                  "working_dir": relative(run_dir)},
    "versions": versions, "public_leaderboard_score": None, "error": None,
})
write_once(test_dir / f"{RUN_ID}_test.json", {
    "run_id": f"{RUN_ID}_test", "experiment": "07_hrnet_w32", "status": "test_predicted",
    "source_run_id": RUN_ID, "source_checkpoint_sha256": results["checkpoint_sha256"],
    "test_images": test_rows_count, "transform": transform + "; no augmentation",
    "logits_csv": relative(test_dir / f"{RUN_ID}_test_logits.csv"),
    "git_commit": run_record["git_revision"], "device": run_record["gpu"], "versions": versions,
})
if IN_COLAB:
    for folder in (val_dir, test_dir):
        shutil.copytree(folder, DRIVE_OUTPUT / RUN_ID / "experiments/exp07/results" / folder.name,
                        ignore=shutil.ignore_patterns("checkpoints"), dirs_exist_ok=True)
print(f"Fold-0 macro F1 at {THRESHOLD}: {at_threshold['macro_f1']:.5f} (re-scored from {relative(val_csv)})")
print("Saved:", relative(val_dir), "and", relative(test_dir))


Fold-0 macro F1 at 0.25: 0.99733 (re-scored from experiments/exp07/results/hrnet_w32_finetune_fold0_seed42/hrnet_w32_finetune_fold0_seed42_val.csv)
Saved: experiments/exp07/results/hrnet_w32_finetune_fold0_seed42 and experiments/exp07/results/hrnet_w32_finetune_fold0_seed42_test


## Submission file

Threshold the saved test probabilities at the fixed 0.25 and write the 0/1 CSV in `sample_submission.csv` order, like the other single-model submissions. The raw probability and logit files above are kept unchanged for the ensemble. An existing CSV is never overwritten. Upload to Kaggle by hand.


In [9]:
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from stage2.experiment import LABEL_COLUMNS, form_submission, write_submission

test_file = run_dir / results["files"]["test"]["probabilities"]["path"]
if hashlib.sha256(test_file.read_bytes()).hexdigest() != results["files"]["test"]["probabilities"]["sha256"]:
    raise ValueError(f"{test_file} differs from the exported checkpoint's predictions")
with test_file.open(newline="") as stream:
    rows = list(csv.DictReader(stream))
test_ids = [row["image_id"] for row in rows]
test_probabilities = np.asarray([[float(row[label]) for label in LABEL_COLUMNS] for row in rows])

submission_path = WORK_ROOT / "submissions" / f"exp07_hrnet_w32_fold0_seed42_{round(THRESHOLD * 100):03d}.csv"
if submission_path.exists():
    print("Submission already exists, not overwritten:", submission_path)
else:
    write_submission(submission_path, form_submission(WORK_ROOT / "data/sample_submission.csv", test_ids, test_probabilities, THRESHOLD))
    print("Submission written:", submission_path)
print("Positives per label at", THRESHOLD, {label: int((test_probabilities[:, i] >= THRESHOLD).sum()) for i, label in enumerate(LABEL_COLUMNS)})


Submission written: /Users/allakovalcuk/AI301/deep-learning-assignment2/submissions/exp07_hrnet_w32_fold0_seed42_025.csv
Positives per label at 0.25 {'Cat': 7828, 'Dog': 7655, 'Moris': 158, 'Motya': 135, 'Biatrix': 179}


In [11]:
from stage2.ensemble import mean_probabilities, read_probabilities

MEMBER_RUNS = {  # name -> (experiment folder, run id) of the saved fold-0 runs
    "Exp1 ConvNeXt": ("exp1", "convnext_standard_fold0_seed42_20261018_full1"),
    "Exp2 ConvNeXt+block": ("exp2", "convnext_custom_fold0_seed42_mps_gradcam3"),
    "Exp3 EfficientNetV2-S": ("exp3", "efficientnetv2s_fold0_seed42"),
    "Exp5 BiT": ("exp5", "bit_r50x1_fold0_seed42"),
}
hrnet = test_probabilities >= THRESHOLD
others, missing = {}, []
for name, (folder, member_run) in MEMBER_RUNS.items():
    path = WORK_ROOT / "experiments" / folder / "results" / f"{member_run}_test" / f"{member_run}_test.csv"
    if path.is_file():
        others[name] = read_probabilities(path, test_ids)
    else:
        missing.append(name)
if missing:
    print("Missing test predictions, skipped:", missing)
members = {name: others[name] for name in ("Exp1 ConvNeXt", "Exp3 EfficientNetV2-S", "Exp5 BiT") if name in others}
if len(members) == 3:
    others["Ensemble Exp1+3+5 mean (0.98047)"] = mean_probabilities(list(members.values()))

named = LABEL_COLUMNS[2:]
header = f"{'HRNet vs':<34} {'cells':>6} {'images':>7} " + " ".join(f"{label:>8}" for label in LABEL_COLUMNS)
print(f"HRNet positives at {THRESHOLD}:", {label: int(hrnet[:, i].sum()) for i, label in enumerate(LABEL_COLUMNS)})
print(f"Cells = differing image-label decisions (of {hrnet.size:,}); images = images with at least one differing label.\n")
print(header)
print("-" * len(header))
for name, probabilities in others.items():
    changed = hrnet != (probabilities >= THRESHOLD)
    print(f"{name:<34} {int(changed.sum()):>6} {int(changed.any(axis=1).sum()):>7} " + " ".join(f"{int(changed[:, i].sum()):>8}" for i in range(len(LABEL_COLUMNS))))
print("\nNamed-pet cells that differ, as 'HRNet yes, other no / HRNet no, other yes':")
for name, probabilities in others.items():
    decided = probabilities >= THRESHOLD
    print(f"  {name:<34} " + "  ".join(f"{label} {int((hrnet & ~decided)[:, LABEL_COLUMNS.index(label)].sum())}/{int((~hrnet & decided)[:, LABEL_COLUMNS.index(label)].sum())}" for label in named))
pairs = [("Exp1 ConvNeXt", "Exp3 EfficientNetV2-S"), ("Exp3 EfficientNetV2-S", "Exp5 BiT"), ("Exp1 ConvNeXt", "Exp5 BiT")]
print("\nFor reference, earlier members against each other (cells):",
      {f"{a.split()[0]}-{b.split()[0]}": int(((others[a] >= THRESHOLD) != (others[b] >= THRESHOLD)).sum()) for a, b in pairs if a in others and b in others})


HRNet positives at 0.25: {'Cat': 7828, 'Dog': 7655, 'Moris': 158, 'Motya': 135, 'Biatrix': 179}
Cells = differing image-label decisions (of 93,365); images = images with at least one differing label.

HRNet vs                            cells  images      Cat      Dog    Moris    Motya  Biatrix
----------------------------------------------------------------------------------------------
Exp1 ConvNeXt                         283     238      101      104       38       26       14
Exp2 ConvNeXt+block                   281     232      103      104       38       24       12
Exp3 EfficientNetV2-S                 272     224      103      116       17       22       14
Exp5 BiT                              314     253      112      123       27       30       22
Ensemble Exp1+3+5 mean (0.98047)      257     227       99      111       21       16       10

Named-pet cells that differ, as 'HRNet yes, other no / HRNet no, other yes':
  Exp1 ConvNeXt                      Moris 17/21  Motya 

## How different is HRNet from the other members?

An ensemble only helps when its members make different mistakes. Test labels are hidden, so we cannot say who is right, but we can count where HRNet's 0/1 decisions at 0.25 differ from each earlier member's and from our best ensemble (Exp1 + Exp3 + Exp5 mean, public 0.98047). A "cell" is one image-label pair (18,673 images × 5 labels); an "image" differs if any of its five labels does. A named-pet cell is worth roughly 50 times a Cat or Dog cell for macro F1, so the named-pet columns matter most. Nothing is written.


## Result and next step

Public leaderboard score of `exp07_hrnet_w32_fold0_seed42_025.csv`: **0.96431**.

| Model | Fold-0 macro F1 at 0.25 | Public leaderboard |
|---|---:|---:|
| Exp1: ConvNeXt-base | 0.97518 | 0.97003 |
| Exp3: EfficientNetV2-S | 0.99831 | 0.97015 |
| Exp5: BiT ResNetV2-50x1 | 0.99851 | 0.94035 |
| **Exp7: HRNet-W32** (this notebook) | 0.99733 | **0.96431** |

**1. HRNet is a middle member.** It is 0.0058 below Exp3 and Exp1, above Exp2 (0.96043) and well above BiT (0.94035). Its fold-0 score (0.99733) again overstated it: the gap to the leaderboard is 0.033, against 0.005 for Exp1, 0.028 for Exp3 and 0.058 for Exp5. Fold 0 has 8–10 photos per named pet, and the checkpoint was selected on it, so it cannot rank models.

**2. The run.** The planned 9 epochs took 33 minutes each. Epoch 8 had the best fold-0 macro F1. Training loss fell to about 1e-4 while validation loss stayed near 0.007–0.008, the same over-confident pattern as Exp3 and Exp5. We did not tune the learning rates (backbone 3e-5, head 3e-4).

**3. It differs from the other members**, by 272–314 test cells (see the comparison above), but adding it to the Exp1 + Exp3 + Exp5 mean lowered the public score from 0.98047 to 0.97816 (`18_ensemble_exp1_exp3_exp5_exp7.ipynb`). The reasons are in that notebook.

**4. Not tested.** Why HRNet generalizes worse than fold 0 suggests. Candidates: it is the only member pretrained on ImageNet-1k only (the others used ImageNet-21k and then 1k), its native resolution is 224 and we used 384, its linear head was weaker than the other warm-ups (fold-0 macro F1 0.917), and the fold-0 checkpoint selection may have been lucky.

The raw probabilities and logits are in `experiments/exp07/results/` for any further ensemble. Compare HRNet with existing members on the same validation IDs, then tune thresholds locally without retraining. A fold-0 score alone cannot establish an improvement.
